In [1]:
import time, html, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import os

In [2]:
CSV_PATH = "../data/fuel_consumption.csv"
PARQUET_PATH = "../data/fuel_consumption.parquet"

metadata = pq.ParquetFile(PARQUET_PATH).metadata

In [4]:
# Les premiers octets bruts : illisibles tels quels.
with open(PARQUET_PATH, "rb") as f:
    raw_start = f.read(16)


In [26]:

# --- Métadonnées au niveau du fichier ---
print("File metadata :")
print(f"  Version du format Parquet : {metadata.format_version}")
print(f"  Créé par                  : {metadata.created_by}")
print(f"  Nombre de lignes          : {metadata.num_rows}")
print(f"  Nombre de row groups      : {metadata.num_row_groups}")
print(f"  Nombre de colonnes        : {metadata.num_columns}")
print(f"  Taille des métadonnées    : {metadata.serialized_size} octets")
 
# --- Métadonnées au niveau du schéma (incluant celles ajoutées par pandas) ---
print("\n Schema Métadonnées (arrow) :")
print(pq.read_schema(PARQUET_PATH))
if metadata.metadata:
    print("\n Metadata (key/Values) :")
    for key, value in metadata.metadata.items():
        print(f"  {key}: {value[:120]}{'...' if len(value) > 120 else ''}")
 
# --- Métadonnées au niveau des row groups et des colonnes : compression
# et encoding utilisés, concrètement, pour chaque colonne de chaque row
# group (Parquet peut choisir un encoding différent par colonne).
print("\nCompression and encoding by column and row group :")
for rg_index in range(metadata.num_row_groups):
    rg = metadata.row_group(rg_index)
    print(f"\n  Row group {rg_index} ({rg.num_rows} lines, "
          f"{rg.total_byte_size / 1024:.1f} Ko uncompressed) :")
    for col_index in range(rg.num_columns):
        col = rg.column(col_index)
        print(f"    {col.path_in_schema:<20} "
              f"compression={col.compression:<8} "
              f"encodings={col.encodings} "
              f"({col.total_compressed_size / 1024:.1f} Ko compressed / "
              f"{col.total_uncompressed_size / 1024:.1f} Ko row)")
 
 


File metadata :
  Version du format Parquet : 2.6
  Créé par                  : parquet-cpp-arrow version 25.0.1
  Nombre de lignes          : 17766
  Nombre de row groups      : 1
  Nombre de colonnes        : 13
  Taille des métadonnées    : 7136 octets

 Schema Métadonnées (arrow) :
Model year: int64
Make: large_string
Model: large_string
Vehicle class: large_string
Engine size (L): double
Cylinders: int64
Transmission: large_string
Fuel type: large_string
City (L/100 km): double
Highway (L/100 km): double
Combined (L/100 km): double
Combined (mpg): int64
CO2 emissions (g/km): int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1691

 Metadata (key/Values) :
  b'pandas': b'{"index_columns": [], "column_indexes": [], "columns": [{"name": "Model year", "field_name": "Model year", "pandas_type"'...
  b'ARROW:schema': b'/////zAKAAAQAAAAAAAKAA4ABgAFAAgACgAAAAABBAAQAAAAAAAKAAwAAAAEAAgACgAAABAHAAAEAAAAAQAAAAwAAAAIAAwABAAIAAgAAADoBgAABAAA

In [6]:
## Size

In [25]:
csv_size = os.path.getsize(CSV_PATH)
parquet_size = os.path.getsize(PARQUET_PATH)
print("=== Disk size ===")
print(f"CSV     : {csv_size / 1024:.1f} Ko")
print(f"Parquet : {parquet_size / 1024:.1f} Ko")
print(f"Ratio   : {csv_size / parquet_size:.2f}x\n")
 
start = time.perf_counter()
df_csv = pd.read_csv(CSV_PATH)
csv_read_time = time.perf_counter() - start
 
start = time.perf_counter()
df_parquet = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
parquet_read_time = time.perf_counter() - start
 
print("=== Speed (all file) ===")
print(f"CSV     : {csv_read_time:.4f} s")
print(f"Parquet : {parquet_read_time:.4f} s\n")


=== Disk size ===
CSV     : 1244.2 Ko
Parquet : 175.6 Ko
Ratio   : 7.08x

=== Speed (all file) ===
CSV     : 0.0486 s
Parquet : 0.0201 s



In [9]:
# COLUMN PRUNING

In [24]:
row_group = metadata.row_group(0)

start = time.perf_counter()
pd.read_parquet(PARQUET_PATH, engine="pyarrow")
full_read_time = time.perf_counter() - start
 
start = time.perf_counter()
pd.read_parquet(PARQUET_PATH, engine="pyarrow", columns=[ 'Combined (mpg)', 'Make', 'Model' ])
pruned_read_time = time.perf_counter() - start

start = time.perf_counter()
pd.read_csv(CSV_PATH)
csv_full_read_time = time.perf_counter() - start
 
start = time.perf_counter()
pd.read_csv(CSV_PATH, usecols=["Combined (mpg)", "Make", "Model"])
csv_pruned_read_time = time.perf_counter() - start

print(f"\nParquet, read all columns                                       : {full_read_time:.4f} s")
print(f"Parquet, read specific columns (Combined (mpg), Make, Model)    : {pruned_read_time:.4f} s")
print()
print(f"\n CSV, read all columns                                        : {csv_full_read_time:.4f} s")
print(f" CSV, read specific columns (Combined (mpg), Make, Model)     : {csv_pruned_read_time:.4f} s")



Parquet, read all columns                                       : 0.0226 s
Parquet, read specific columns (Combined (mpg), Make, Model)    : 0.0142 s


 CSV, read all columns                                        : 0.0517 s
 CSV, read specific columns (Combined (mpg), Make, Model)     : 0.0621 s


In [ ]:
# PREDICATE PUSHDOWN

In [16]:
FILTER_COLUMN = 'Combined (mpg)'
FILTER_VALUE = 50

metadata = pq.ParquetFile(PARQUET_PATH).metadata

start = time.perf_counter()
df_full = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
df_filtered = df_full[df_full[FILTER_COLUMN] > FILTER_VALUE]
no_pushdown_time = time.perf_counter() - start
 
dataset = ds.dataset(PARQUET_PATH, format="parquet")
start = time.perf_counter()
table_filtered = dataset.to_table(filter=ds.field(FILTER_COLUMN) > FILTER_VALUE)
pushdown_time = time.perf_counter() - start
 
print(f"No pushdown : {no_pushdown_time:.4f} s")
print(f"With pushdown   : {pushdown_time:.4f} s")
print(f"Lignes result : {len(df_filtered)} (pandas) / {table_filtered.num_rows} (pushdown)")

start = time.perf_counter()
df_csv_full = pd.read_csv(CSV_PATH)
df_csv_filtered = df_csv_full[df_csv_full[FILTER_COLUMN] > FILTER_VALUE]
csv_filter_time = time.perf_counter() - start
 
print(f"\nFiltre sur CSV : {csv_filter_time:.4f} s")
print(f"Lignes résultat : {len(df_csv_filtered)} (csv)")

 
print(f"\nStatistic per row group on '{FILTER_COLUMN}' :")
col_index = metadata.schema.names.index(FILTER_COLUMN)
for i in range(metadata.num_row_groups):
    stats = metadata.row_group(i).column(col_index).statistics
    if stats is not None:
        skip = stats.max < FILTER_VALUE
        print(f"  Row group {i}: min={stats.min}, max={stats.max} "
              f"-> {'ignoré' if skip else 'lu'}")



4. PREDICATE PUSHDOWN
No pushdown : 0.0255 s
With pushdown   : 0.0091 s
Lignes result : 159 (pandas) / 159 (pushdown)

Filtre sur CSV : 0.0539 s
Lignes résultat : 159 (csv)

Statistic per row group on 'Combined (mpg)' :
  Row group 0: min=11, max=78 -> lu


In [ ]:
# """
# Conversion d'un fichier CSV en fichier Parquet.
# Le fichier source doit se trouver dans le sous-dossier data/.
# """

# import pandas as pd

# # --- Lecture du CSV ---
# df = pd.read_csv(CSV_PATH)

# # --- Écriture en Parquet ---
# # engine="pyarrow" : moteur utilisé pour l'écriture
# # compression="snappy" : compression par défaut, bon compromis vitesse/taille
# # index=False : on n'écrit pas l'index pandas dans le fichier
# df.to_parquet(PARQUET_PATH, engine="pyarrow", compression="snappy", index=False)

# print(f"Fichier converti : {CSV_PATH} -> {PARQUET_PATH}")